In [1]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:

import pandas as pd

file_path = (
    '/content/drive/MyDrive/CareLoop_NoShow_Guard/'
    'data/followup_notes.csv'
)

notes_df = pd.read_csv(file_path)

print("Dataset shape:", notes_df.shape)
print("\nColumn names:")
print(notes_df.columns.tolist())

print("\nFirst 5 rows:")
display(notes_df.head())


Dataset shape: (30, 3)

Column names:
['note_id', 'appointment_id', 'note_text']

First 5 rows:


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."


In [3]:

!pip -q install -U google-genai pydantic


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.0 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [4]:

%pip install -q "google-auth==2.49.0" "google-genai>=2.12.1,<3" pydantic


In [5]:

from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say: Gemini connection successful!"
)

print(response.text)


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://aistudio.google.com/docs/get-started).', 'status': 'NOT_FOUND'}}

In [ ]:

from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say: Gemini connection successful!"
)

print(response.text)


In [ ]:

import time
from google.genai import errors

for attempt in range(5):
    try:
        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents="Say: Gemini connection successful!"
        )

        print(response.text)
        break

    except errors.ServerError as e:
        if e.code == 503 and attempt < 4:
            wait = 2 ** (attempt + 1)
            print(f"Server busy. Retrying in {wait} seconds...")
            time.sleep(wait)
        else:
            print("Request failed:", e)
            break


In [ ]:

# Create a working copy
notes_clean = notes_df.copy()

# Replace missing values with empty strings
notes_clean['note_text'] = (
    notes_clean['note_text']
    .fillna('')
    .astype(str)
    .str.strip()
)

# Identify notes containing actual text
notes_clean['has_text'] = (
    notes_clean['note_text'] != ''
)

print("Total notes:", len(notes_clean))
print("Notes with text:", notes_clean['has_text'].sum())
print("Blank notes:", (~notes_clean['has_text']).sum())

print("\nFirst 10 notes:")
display(
    notes_clean[
        ['note_id', 'note_text', 'has_text']
    ].head(10)
)


In [ ]:

from pydantic import BaseModel, Field
from typing import Literal

class NoteAnalysis(BaseModel):
    reason: Literal[
        "forgot_or_no_reminder",
        "distance_or_transport",
        "cost_or_fee",
        "illness_or_emergency",
        "work_or_schedule",
        "other"
    ]
    summary: str
    recommended_action: str
    confidence: float = Field(ge=0, le=1)

print("Pydantic schema created successfully!")


In [ ]:

from google.genai import types

# Select the first non-blank patient note
first_note = notes_clean.loc[
    notes_clean['has_text'], 'note_text'
].iloc[0]

prompt = f"""
Analyze this patient appointment follow-up note.

Identify the main reason for the missed appointment.

Choose one reason from:
- forgot_or_no_reminder
- distance_or_transport
- cost_or_fee
- illness_or_emergency
- work_or_schedule
- other

Provide:
1. reason
2. summary (one short sentence)
3. recommended_action (recommended_action:
Suggest ONE preventive action that directly addresses
the root cause of the missed appointment.

Focus on reducing FUTURE no-shows.

If the patient missed an appointment because they
forgot or did not receive a reminder, prioritize
automated reminders and delivery verification.

Do NOT recommend rescheduling as the primary action.

)

Use only information supported by the note.
Do not invent missing details.

Patient note:
{first_note}
"""

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=NoteAnalysis
    )
)

# Validate Gemini's response using Pydantic
analysis = NoteAnalysis.model_validate_json(
    response.text
)

print("Original note:", first_note)
print("\nAI ANALYSIS")
print("Reason:", analysis.reason)
print("Summary:", analysis.summary)
print("Recommended action:", analysis.recommended_action)
print("Confidence:", analysis.confidence)


In [ ]:

import time
from google.genai import errors, types

for attempt in range(5):
    try:
        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=NoteAnalysis
            )
        )

        # Validate AI output
        analysis = NoteAnalysis.model_validate_json(
            response.text
        )

        print("Original note:", first_note)
        print("\nAI ANALYSIS")
        print("Reason:", analysis.reason)
        print("Summary:", analysis.summary)
        print("Recommended action:", analysis.recommended_action)
        print("Confidence:", analysis.confidence)

        break

    except errors.ServerError as e:
        if e.code == 503 and attempt < 4:
            wait = 2 ** (attempt + 1)
            print(f"Server busy. Retrying in {wait} seconds...")
            time.sleep(wait)
        else:
            print("Request failed:", e)
            break


In [ ]:

# List models available to your API key
for model in client.models.list():
    if "flash" in model.name.lower():
        print(model.name)


In [ ]:

from google.genai import types

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=NoteAnalysis
    )
)

analysis = NoteAnalysis.model_validate_json(
    response.text
)

print("Original note:", first_note)
print("\nAI ANALYSIS")
print("Reason:", analysis.reason)
print("Summary:", analysis.summary)
print("Recommended action:", analysis.recommended_action)
print("Confidence:", analysis.confidence)


In [ ]:

import time
import pandas as pd
from google.genai import errors, types

results = []

# Process the first 10 notes
for _, row in notes_clean.head(10).iterrows():

    note_text = row['note_text']

    result = {
        'note_id': row['note_id'],
        'appointment_id': row['appointment_id'],
        'reason': None,
        'summary': None,
        'recommended_action': None,
        'confidence': None,
        'validation_status': None,
        'raw_response': None
    }

    # Skip blank notes
    if not row['has_text']:
        result['validation_status'] = 'skipped_blank'
        results.append(result)
        continue

    note_prompt = f"""
    Analyze this patient appointment follow-up note.

    Identify the main reason for the missed appointment.

    Choose one reason from:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    Return:
    - reason
    - summary: one short sentence
    - recommended_action: one practical action
    - confidence: number between 0 and 1

    Use only information supported by the note.
    Do not invent missing details.

    Patient note:
    {note_text}
    """

    # Retry temporary server failures
    for attempt in range(5):
        try:
            response = client.models.generate_content(
                model="gemini-3.1-flash-lite",
                contents=note_prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=NoteAnalysis
                )
            )

            result['raw_response'] = response.text

            # Validate response with Pydantic
            analysis = NoteAnalysis.model_validate_json(
                response.text
            )

            result.update(analysis.model_dump())
            result['validation_status'] = 'valid'
            break

        except errors.ServerError as e:
            if e.code in (500, 502, 503, 504) and attempt < 4:
                time.sleep(2 ** (attempt + 1))
            else:
                result['validation_status'] = 'api_error'
                break

        except Exception as e:
            result['validation_status'] = 'error'
            print("Error on note:", row['note_id'], str(e)[:150])
            break

    results.append(result)
    print("Processed note:", row['note_id'],
          "| Status:", result['validation_status'])

    # Small delay between requests
    time.sleep(1)

# Create DataFrame
notes_analysed = pd.DataFrame(results)

print("\nDataset shape:", notes_analysed.shape)
print("\nValidation status:")
print(notes_analysed['validation_status'].value_counts())

display(notes_analysed[
    ['note_id', 'reason', 'confidence', 'validation_status']
])


In [ ]:

# Review all successfully analysed notes
display(
    notes_analysed.loc[
        notes_analysed['validation_status'] == 'valid',
        ['note_id', 'reason', 'recommended_action']
    ]
)


In [ ]:

output_path = (
    '/content/drive/MyDrive/CareLoop_NoShow_Guard/'
    'notes_analysed.csv'
)

notes_analysed.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Rows:", len(notes_analysed))
print("Columns:", len(notes_analysed.columns))


In [ ]:

# Load cleaned appointment data
appointments = pd.read_csv(
    '/content/drive/MyDrive/CareLoop_NoShow_Guard/'
    'appointments_clean.csv'
)

# Join Gemini analysis with appointments
notes_joined = notes_analysed.merge(
    appointments[
        ['appointment_id', 'no_show', 'lead_days']
    ],
    on='appointment_id',
    how='left',
    validate='many_to_one'
)

print("Joined dataset shape:", notes_joined.shape)

print("\nNo-show distribution:")
print(notes_joined['no_show'].value_counts(dropna=False))

print("\nReasons identified by Gemini:")
print(notes_joined['reason'].value_counts(dropna=False))

display(
    notes_joined[
        ['note_id', 'reason', 'no_show', 'lead_days']
    ]
)


In [ ]:

# Load cleaned appointment data
appointments = pd.read_csv(
    '/content/drive/MyDrive/CareLoop_NoShow_Guard/'
    'appointments_clean.csv'
)

# Join Gemini analysis with appointments
notes_joined = notes_analysed.merge(
    appointments[
        ['appointment_id', 'no_show', 'lead_days']
    ],
    on='appointment_id',
    how='left',
    validate='many_to_one'
)

print("Joined dataset shape:", notes_joined.shape)

print("\nNo-show distribution:")
print(notes_joined['no_show'].value_counts(dropna=False))

print("\nReasons identified by Gemini:")
print(notes_joined['reason'].value_counts(dropna=False))

display(
    notes_joined[
        ['note_id', 'reason', 'no_show', 'lead_days']
    ]
)


In [ ]:

reason_summary = (
    notes_joined[
        notes_joined['validation_status'] == 'valid'
    ]
    .groupby('reason')
    .agg(
        total_notes=('note_id', 'count'),
        avg_lead_days=('lead_days', 'mean')
    )
    .reset_index()
    .sort_values('total_notes', ascending=False)
)

reason_summary['avg_lead_days'] = (
    reason_summary['avg_lead_days'].round(1)
)

print("GEMINI REASON SUMMARY")
display(reason_summary)


In [ ]:

reason_summary = (
    notes_joined[
        notes_joined['validation_status'] == 'valid'
    ]
    .groupby('reason')
    .agg(
        total_notes=('note_id', 'count'),
        avg_lead_days=('lead_days', 'mean')
    )
    .reset_index()
    .sort_values('total_notes', ascending=False)
)

reason_summary['avg_lead_days'] = (
    reason_summary['avg_lead_days'].round(1)
)

print("GEMINI REASON SUMMARY")
display(reason_summary)


In [ ]:

import os

project_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard"

print("PROJECT FILES")
print("-------------")

for root, dirs, files in os.walk(project_path):
    level = root.replace(project_path, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in sorted(files):
        print(f"{indent}  {file}")


In [ ]:

project_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard"

requirements = """pandas
numpy
requests
scikit-learn
pydantic
google-genai>=2.12.1,<3
joblib
"""

with open(
    f"{project_path}/requirements.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(requirements)

print("requirements.txt created successfully!")


In [ ]:

import os

project_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard"

gitignore_content = """# API keys and environment variables
.env
.env.*
!.env.example
*credentials*.json
*api_key*.txt

# Python cache
__pycache__/
*.pyc

# Jupyter temporary files
.ipynb_checkpoints/

# Virtual environments
venv/
.venv/

# Operating system files
.DS_Store
Thumbs.db
"""

gitignore_path = os.path.join(
    project_path, ".gitignore"
)

with open(gitignore_path, "w", encoding="utf-8") as f:
    f.write(gitignore_content)

print(".gitignore created:", os.path.exists(gitignore_path))


In [ ]:

import os

project_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard"

readme = """P# CareLoop Clinics – No-Show Guard

### End-to-End Data Analytics, Machine Learning & Generative AI Project

An end-to-end healthcare analytics project designed to identify patterns in missed outpatient appointments, predict no-show risk, and analyze patient follow-up notes using Generative AI.

## Project Objectives

- Clean and validate appointment booking data.
- Analyze no-show patterns using SQL.
- Build machine learning models to predict appointment no-shows.
- Identify high-risk appointments for targeted reminders.
- Extract missed-appointment reasons and preventive recommendations using Gemini AI.

## Tech Stack

**Languages:** Python, SQL

**Libraries:** Pandas, NumPy, Scikit-learn, Pydantic, Requests, Joblib

**Tools:** Google Colab, MySQL, Google Gemini API, GitHub

## Dataset

- Original appointment records: 1,020
- Cleaned appointment records: 990
- Clinics: 8
- No-show appointments: 218
- Overall no-show rate: 22.02%
- Patient follow-up notes: 30

## Project Workflow

### 1. Data Ingestion and Cleaning

- Loaded CSV datasets using Python.
- Removed duplicate and invalid appointment records.
- Handled missing values and outliers.
- Engineered booking lead-time and no-show target features.
- Enriched clinic information using the Open-Meteo API.

### 2. SQL Analysis

Used MySQL joins, aggregations, CTEs, window functions, and ranking functions to analyze appointment trends.

**Key finding:** Appointments booked 31 or more days in advance had a 60.5% no-show rate, compared with 16.2% for appointments booked 1–7 days in advance.

### 3. Machine Learning

Trained and evaluated:

- Logistic Regression
- Decision Tree
- Random Forest

Selected Logistic Regression based on cross-validation performance and tuned the classification threshold using training data.

**Final held-out test performance:**

| Metric | Result |
|---|---:|
| Accuracy | 79.3% |
| Precision | 57.1% |
| Recall | 65.3% |
| F1-score | 0.610 |
| ROC-AUC | 0.789 |
| Classification threshold | 0.30 |

The final model identified 32 of 49 actual no-shows in the held-out test set.

### 4. Generative AI Integration

Integrated the Gemini API with Pydantic validation to:

- Categorize reasons for missed appointments.
- Generate concise summaries.
- Recommend preventive actions.
- Validate structured AI responses.
- Handle blank notes and temporary API failures.

Processed 10 follow-up notes, including 9 valid AI analyses and 1 skipped blank note.

## Key Insights

- Longer booking lead times were associated with higher no-show rates.
- Logistic Regression achieved the strongest cross-validation ROC-AUC among the evaluated model configurations.
- Reducing the classification threshold from 0.50 to 0.30 improved test recall from 34.7% to 65.3%.
- Follow-up note analysis identified transport difficulties, reminder issues, and scheduling conflicts among the reported reasons.

## Limitations

- Results are based on a limited appointment dataset and have not been validated in a live clinical setting.
- The Gemini reason analysis covers only nine nonblank notes.
- AI-generated recommendations require human review before operational use.
- Model performance may change across clinics, populations, or time periods.

## Future Improvements

- Develop an interactive risk-monitoring dashboard.
- Integrate automated reminder workflows.
- Monitor model performance and data drift.
- Evaluate the model using additional clinic data.

## Author

Independent Data Analytics & Machine Learning Capstone Project"""

readme_path = os.path.join(project_path, "README.md")

with open(readme_path, "w", encoding="utf-8") as f:
    f.write(readme)

print("README created:", os.path.exists(readme_path))


In [ ]:

import os
import json
import re

project_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard"

patterns = [
    r"AIza[0-9A-Za-z_-]{30,}",
    r"""(?i)(?:api_key|GEMINI_API_KEY)\s*=\s*["'][^"']+["']"""
]

issues = []

for filename in os.listdir(project_path):
    if filename.endswith(".ipynb"):
        filepath = os.path.join(project_path, filename)

        with open(filepath, "r", encoding="utf-8") as f:
            notebook = json.load(f)

        for cell_number, cell in enumerate(
            notebook.get("cells", []), start=1
        ):
            content = "".join(cell.get("source", []))

            if cell.get("cell_type") == "code":
                for output in cell.get("outputs", []):
                    content += json.dumps(output)

            if any(re.search(p, content) for p in patterns):
                issues.append((filename, cell_number))

if issues:
    print("POTENTIAL SECRETS FOUND:")
    for filename, cell in issues:
        print(f"{filename} - Cell {cell}")
else:
    print("No obvious hardcoded API keys detected.")


In [ ]:

import pandas as pd

project_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard"

files_to_check = [
    "data/appointments.csv",
    "data/clinics.csv",
    "data/followup_notes.csv",
    "appointments_clean.csv",
    "notes_analysed.csv",
    "test_risk_scores.csv"
]

for filename in files_to_check:
    df = pd.read_csv(f"{project_path}/{filename}")
    print(f"\n{filename}")
    print("Columns:", df.columns.tolist())
    print("Rows:", len(df))


In [ ]:

from pathlib import Path

readme_path = Path(
    "/content/drive/MyDrive/CareLoop_NoShow_Guard/README.md"
)

content = readme_path.read_text(encoding="utf-8")

print("README length:", len(content), "characters")
print("Contains placeholder:", "PASTE YOUR README CONTENT HERE" in content)
print("\nFirst 300 characters:\n")
print(content[:300])


In [ ]:

from pathlib import Path

readme_path = Path(
    "/content/drive/MyDrive/CareLoop_NoShow_Guard/README.md"
)

content = readme_path.read_text(encoding="utf-8")

content = content.replace(
    "P# CareLoop Clinics",
    "# CareLoop Clinics",
    1
)

readme_path.write_text(content, encoding="utf-8")

print(content[:55])


In [ ]:

from pathlib import Path
import zipfile

project = Path("/content/drive/MyDrive/CareLoop_NoShow_Guard")
zip_path = Path("/content/CareLoop_GitHub.zip")

files_to_upload = [
    "01_ingest_clean.ipynb",
    "02_queries.sql",
    "03_model.ipynb",
    "04_genai.ipynb",
    "README.md",
    "requirements.txt",
    ".gitignore"
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for filename in files_to_upload:
        file_path = project / filename

        if file_path.exists():
            z.write(file_path, arcname=filename)
            print("Added:", filename)
        else:
            print("MISSING:", filename)

print("\nZIP created:", zip_path.exists())


In [ ]:

from google.colab import files

files.download("/content/CareLoop_GitHub.zip")


In [6]:

import pandas as pd

project_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard"

notes_path = f"{project_path}/notes_analysed.csv"

notes = pd.read_csv(notes_path)

# Rename column to match assignment
notes = notes.rename(columns={
    "reason": "root_cause"
})

# Arrange columns in required order
required_columns = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

notes = notes[required_columns]

notes.to_csv(notes_path, index=False)

print("Updated columns:")
print(notes.columns.tolist())

print("\nValidation status:")
print(notes["validation_status"].value_counts())

print("\nBlank note:")
print(notes.loc[
    notes["validation_status"] == "skipped_blank",
    ["note_id", "root_cause", "confidence",
     "summary", "recommended_action"]
])


Updated columns:
['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response']

Validation status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

Blank note:
  note_id root_cause  confidence summary recommended_action
3    N004        NaN         NaN     NaN                NaN


In [7]:

import pandas as pd

project_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard"
notes_path = f"{project_path}/notes_analysed.csv"

notes = pd.read_csv(notes_path)

# Find the blank note
blank_mask = notes["note_id"] == "N004"

# Set assignment-required defaults
notes.loc[blank_mask, "root_cause"] = "other"
notes.loc[blank_mask, "confidence"] = 0.0
notes.loc[blank_mask, "summary"] = "No note recorded"
notes.loc[blank_mask, "recommended_action"] = "Re-call patient"
notes.loc[blank_mask, "validation_status"] = "skipped_blank"
notes.loc[blank_mask, "raw_response"] = ""

# Save corrected file
notes.to_csv(notes_path, index=False)

print("N004 after correction:")
display(notes.loc[blank_mask])
print("\nTotal rows:", len(notes))


N004 after correction:


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank,



Total rows: 10


In [8]:

from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

# Test 1: Confidence exceeds 1
invalid_confidence = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment.",
    "recommended_action": "Send automated reminders."
}

# Test 2: Missing recommended_action
missing_action = {
    "root_cause": "work_or_schedule",
    "confidence": 0.8,
    "summary": "Patient had a work conflict."
}

for test_name, test_data in [
    ("Invalid confidence", invalid_confidence),
    ("Missing recommended_action", missing_action)
]:
    try:
        FollowupInsight(**test_data)
        print(test_name, "- Unexpectedly passed!")

    except ValidationError as e:
        print(f"\n{test_name} - Correctly rejected!")
        print(e.errors()[0]["msg"])



Invalid confidence - Correctly rejected!
Input should be less than or equal to 1

Missing recommended_action - Correctly rejected!
Field required


In [9]:

import json
from google.genai import types

allowed_causes = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

# Use a plain string, as required by the assignment
comparison_schema = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause", "confidence",
        "summary", "recommended_action"
    ]
}

# Load the original notes
notes_original = pd.read_csv(
    "/content/drive/MyDrive/CareLoop_NoShow_Guard/data/followup_notes.csv"
)

comparison_results = []

for note_id in ["N007", "N009"]:
    note_text = notes_original.loc[
        notes_original["note_id"] == note_id, "note_text"
    ].iloc[0]

    prompts = {
        "V1": f"Classify this note:\n{note_text}",
        "V2": f"""
Context: You are reviewing missed clinic appointments.
Role: Act as a healthcare operations analyst.
Action: Identify the primary reason for the missed visit.
Allowed root_cause values: {', '.join(allowed_causes)}.
If the reason is unclear, use 'other' rather than guessing.
Tone: Clear, factual, and concise.
Format: Return root_cause, confidence, summary,
and recommended_action as JSON.

Patient note: {note_text}
"""
    }

    for version, prompt in prompts.items():
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0,
                response_mime_type="application/json",
                response_schema=comparison_schema,
                max_output_tokens=1000
            )
        )

        result = json.loads(response.text)

        comparison_results.append({
            "note_id": note_id,
            "prompt_version": version,
            "root_cause": result.get("root_cause"),
            "allowed_category": result.get("root_cause") in allowed_causes
        })

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)


,note_id,prompt_version,root_cause,allowed_category
0,N007,V1,User did not receive the booking confirmation ...,False
1,N007,V2,forgot_or_no_reminder,True
2,N009,V1,Patient unreachable due to phone being powered...,False
3,N009,V2,other,True


In [10]:

project_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard"

comparison_df.to_csv(
    f"{project_path}/prompt_comparison.csv",
    index=False
)

print("Prompt comparison saved successfully!")
print("Successful analysis calls in final workflow: 14")
print("Note: Earlier experiments and retries are not included.")


Prompt comparison saved successfully!
Successful analysis calls in final workflow: 14
Note: Earlier experiments and retries are not included.


In [11]:

import pandas as pd

project_path = "/content/drive/MyDrive/CareLoop_NoShow_Guard"

# Load Gemini analysis
notes = pd.read_csv(
    f"{project_path}/notes_analysed.csv"
)

# Load cleaned appointments
appointments = pd.read_csv(
    f"{project_path}/appointments_clean.csv"
)

# Join on appointment ID
notes_merged = notes.merge(
    appointments[
        ["appointment_id", "reminder_sent", "no_show"]
    ],
    on="appointment_id",
    how="left",
    validate="many_to_one"
)

# Cross-tabulate reasons against reminders
reason_reminder_table = pd.crosstab(
    notes_merged["root_cause"],
    notes_merged["reminder_sent"]
)

print("MERGED DATASET SHAPE:")
print(notes_merged.shape)

print("\nREASONS VS REMINDERS:")
display(reason_reminder_table)

print("\nTOTAL NOTES:", len(notes_merged))
print("NOTES WITHOUT REMINDERS:",
      (notes_merged["reminder_sent"] == "No").sum())


MERGED DATASET SHAPE:
(10, 10)

REASONS VS REMINDERS:


reminder_sent,No,Yes
root_cause,,
cost_or_fee,0,1
distance_or_transport,1,1
forgot_or_no_reminder,2,0
illness_or_emergency,0,1
other,1,1
work_or_schedule,1,1



TOTAL NOTES: 10
NOTES WITHOUT REMINDERS: 5


In [12]:

# Save merged data for reproducibility
notes_merged.to_csv(
    f"{project_path}/notes_merged.csv",
    index=False
)

# Save cross-tabulation
reason_reminder_table.to_csv(
    f"{project_path}/reason_reminder_crosstab.csv"
)

print("Part 4.4 files saved successfully!")

print("\nKEY FINDINGS:")
print("Total analysed notes:", len(notes_merged))
print(
    "Appointments without reminders:",
    (notes_merged["reminder_sent"] == "No").sum()
)
print(
    "Forgot/no-reminder cases:",
    (notes_merged["root_cause"] == "forgot_or_no_reminder").sum()
)


Part 4.4 files saved successfully!

KEY FINDINGS:
Total analysed notes: 10
Appointments without reminders: 5
Forgot/no-reminder cases: 2


In [13]:

from pathlib import Path

project_path = Path("/content/drive/MyDrive/CareLoop_NoShow_Guard")
readme_path = project_path / "README.md"

readme = readme_path.read_text(encoding="utf-8")

print("README length:", len(readme))
print("\nCURRENT README:\n")
print(readme)


README length: 3516

CURRENT README:

# CareLoop Clinics – No-Show Guard

### End-to-End Data Analytics, Machine Learning & Generative AI Project

An end-to-end healthcare analytics project designed to identify patterns in missed outpatient appointments, predict no-show risk, and analyze patient follow-up notes using Generative AI.

## Project Objectives

- Clean and validate appointment booking data.
- Analyze no-show patterns using SQL.
- Build machine learning models to predict appointment no-shows.
- Identify high-risk appointments for targeted reminders.
- Extract missed-appointment reasons and preventive recommendations using Gemini AI.

## Tech Stack

**Languages:** Python, SQL

**Libraries:** Pandas, NumPy, Scikit-learn, Pydantic, Requests, Joblib

**Tools:** Google Colab, MySQL, Google Gemini API, GitHub

## Dataset

- Original appointment records: 1,020
- Cleaned appointment records: 990
- Clinics: 8
- No-show appointments: 218
- Overall no-show rate: 22.02%
- Patient follow-

In [14]:

from pathlib import Path

project_path = Path("/content/drive/MyDrive/CareLoop_NoShow_Guard")
readme_path = project_path / "README.md"

readme = readme_path.read_text(encoding="utf-8")

# Replace outdated test metrics
readme = readme.replace(
    "| Accuracy | 79.3% |",
    "| Accuracy | 78.8% |"
)

readme = readme.replace(
    "| Recall | 65.3% |",
    "| Recall | 57.1% |"
)

readme = readme.replace(
    "| F1-score | 0.610 |",
    "| F1-score | 0.571 |"
)

readme = readme.replace(
    "| Classification threshold | 0.30 |",
    "| Classification threshold | 0.35 |"
)

# Correct the number of detected no-shows
readme = readme.replace(
    "The final model identified 32 of 49 actual no-shows in the held-out test set.",
    "The final model identified 28 of 49 actual no-shows in the held-out test set, with 49 appointments flagged as high risk."
)

# Correct the threshold-related insight
readme = readme.replace(
    "Reducing the classification threshold from 0.50 to 0.30 improved test recall from 34.7% to 65.3%.",
    "Using a classification threshold of 0.35, selected through training cross-validation, achieved 57.1% recall and 0.789 ROC-AUC on the held-out test set."
)

readme_path.write_text(readme, encoding="utf-8")

print("README ML metrics updated successfully!")

print("\nUPDATED TEST RESULTS SECTION:\n")
start = readme.index("**Final held-out test performance:**")
end = readme.index("### 4. Generative AI Integration")
print(readme[start:end])


README ML metrics updated successfully!

UPDATED TEST RESULTS SECTION:

**Final held-out test performance:**

| Metric | Result |
|---|---:|
| Accuracy | 78.8% |
| Precision | 57.1% |
| Recall | 57.1% |
| F1-score | 0.571 |
| ROC-AUC | 0.789 |
| Classification threshold | 0.35 |

The final model identified 28 of 49 actual no-shows in the held-out test set, with 49 appointments flagged as high risk.




In [15]:

from pathlib import Path

readme_path = Path(
    "/content/drive/MyDrive/CareLoop_NoShow_Guard/README.md"
)

readme = readme_path.read_text(encoding="utf-8")

new_section = """## Key Insights

1. **Data Cleaning:** Of 1,020 original appointments, 990 were retained after cleaning. The overall no-show rate was 22.02% (218 appointments).

2. **SQL Analysis:** Appointments booked 31 or more days in advance had a 60.5% no-show rate (26 of 43), compared with 16.2% (63 of 390) for bookings made 1–7 days in advance.

3. **Machine Learning:** Logistic Regression achieved 0.789 ROC-AUC on the held-out test set. At the training-selected threshold of 0.35, the model identified 28 of 49 actual no-shows, flagging 49 appointments.

4. **Generative AI:** Of 10 follow-up notes, 9 were analysed using Gemini and 1 blank note was handled without an API call. Five appointments had no reminder sent, including both cases classified as forgot_or_no_reminder.

## Business Recommendation

Prioritise proactive reminders and confirmation calls for appointments with long booking lead times and high predicted no-show risk. In the cleaned dataset, appointments booked 31 or more days ahead had a 60.5% no-show rate, compared with 16.2% for those booked 1–7 days ahead. The Logistic Regression model achieved 0.789 test ROC-AUC and flagged 49 appointments at a 0.35 threshold, identifying 28 of 49 actual no-shows. In the small follow-up sample, 5 of 10 appointments had no reminder sent, and both forgot_or_no_reminder cases lacked reminders. Clinics should first pilot targeted reminders, then monitor attendance outcomes before expanding the intervention.

"""

start = readme.index("## Key Insights")
end = readme.index("## Limitations")

readme = readme[:start] + new_section + readme[end:]

readme_path.write_text(readme, encoding="utf-8")

print("README insights and recommendation updated!")
print("\n")
print(readme[start:readme.index("## Limitations")])


README insights and recommendation updated!


## Key Insights

1. **Data Cleaning:** Of 1,020 original appointments, 990 were retained after cleaning. The overall no-show rate was 22.02% (218 appointments).

2. **SQL Analysis:** Appointments booked 31 or more days in advance had a 60.5% no-show rate (26 of 43), compared with 16.2% (63 of 390) for bookings made 1–7 days in advance.

3. **Machine Learning:** Logistic Regression achieved 0.789 ROC-AUC on the held-out test set. At the training-selected threshold of 0.35, the model identified 28 of 49 actual no-shows, flagging 49 appointments.

4. **Generative AI:** Of 10 follow-up notes, 9 were analysed using Gemini and 1 blank note was handled without an API call. Five appointments had no reminder sent, including both cases classified as forgot_or_no_reminder.

## Business Recommendation

Prioritise proactive reminders and confirmation calls for appointments with long booking lead times and high predicted no-show risk. In the cleaned data

In [16]:

from pathlib import Path

project_path = Path("/content/drive/MyDrive/CareLoop_NoShow_Guard")

required_files = [
    "01_ingest_clean.ipynb",
    "02_queries.sql",
    "03_model.ipynb",
    "04_genai.ipynb",
    "README.md",
    "requirements.txt",
    ".gitignore",
    "appointments_clean.csv",
    "clinics_enriched.csv",
    "rejected_rows.json",
    "load_data.sql",
    "noshow_model.joblib",
    "test_risk_scores.csv",
    "notes_analysed.csv",
    "prompt_comparison.csv",
    "notes_merged.csv",
    "reason_reminder_crosstab.csv",
    "data/appointments.csv",
    "data/clinics.csv",
    "data/followup_notes.csv"
]

print("FINAL PROJECT FILE CHECK\n")

missing_files = []

for file in required_files:
    path = project_path / file

    if path.is_file():
        print(f"OK      {file}")
    else:
        print(f"MISSING {file}")
        missing_files.append(file)

print("\nTotal files checked:", len(required_files))
print("Files found:", len(required_files) - len(missing_files))
print("Missing files:", len(missing_files))


FINAL PROJECT FILE CHECK

OK      01_ingest_clean.ipynb
OK      02_queries.sql
OK      03_model.ipynb
OK      04_genai.ipynb
OK      README.md
OK      requirements.txt
OK      .gitignore
OK      appointments_clean.csv
OK      clinics_enriched.csv
OK      rejected_rows.json
OK      load_data.sql
OK      noshow_model.joblib
OK      test_risk_scores.csv
OK      notes_analysed.csv
OK      prompt_comparison.csv
OK      notes_merged.csv
OK      reason_reminder_crosstab.csv
OK      data/appointments.csv
OK      data/clinics.csv
OK      data/followup_notes.csv

Total files checked: 20
Files found: 20
Missing files: 0


In [17]:

import json
import re
from pathlib import Path

project_path = Path("/content/drive/MyDrive/CareLoop_NoShow_Guard")

notebooks = [
    "01_ingest_clean.ipynb",
    "03_model.ipynb",
    "04_genai.ipynb"
]

patterns = {
    "Google API key pattern": r"AIza[0-9A-Za-z_-]{30,}",
    "Hardcoded API key": r"""(?:api_key|GEMINI_API_KEY)\s*=\s*["'][^"']{15,}["']"""
}

findings = []

for notebook_name in notebooks:
    path = project_path / notebook_name
    notebook = json.loads(path.read_text(encoding="utf-8"))

    for cell_number, cell in enumerate(notebook["cells"], start=1):
        source = "".join(cell.get("source", []))

        for label, pattern in patterns.items():
            if re.search(pattern, source, re.IGNORECASE):
                findings.append((notebook_name, cell_number, label))

print("API KEY SECURITY CHECK\n")

if findings:
    for name, cell, issue in findings:
        print(f"WARNING: {name}, cell {cell}: {issue}")
else:
    print("PASS: No obvious hardcoded API keys found.")

print("\nNote: This checks common patterns, not every possible secret.")


API KEY SECURITY CHECK

PASS: No obvious hardcoded API keys found.

Note: This checks common patterns, not every possible secret.


In [18]:

import json
import re
from pathlib import Path

project_path = Path("/content/drive/MyDrive/CareLoop_NoShow_Guard")

notebooks = [
    "01_ingest_clean.ipynb",
    "03_model.ipynb",
    "04_genai.ipynb"
]

secret_patterns = {
    "Google API key": r"AIza[0-9A-Za-z_-]{30,}",
    "Bearer token": r"Bearer\s+[A-Za-z0-9._-]{20,}"
}

print("NOTEBOOK OUTPUT SECURITY CHECK\n")

issues = []
total_outputs = 0

for name in notebooks:
    notebook = json.loads(
        (project_path / name).read_text(encoding="utf-8")
    )

    output_count = 0

    for cell_number, cell in enumerate(notebook["cells"], start=1):
        for output in cell.get("outputs", []):
            output_count += 1
            output_text = json.dumps(output)

            for label, pattern in secret_patterns.items():
                if re.search(pattern, output_text):
                    issues.append((name, cell_number, label))

    total_outputs += output_count
    print(f"{name}: {output_count} saved outputs")

print("\nTotal saved outputs:", total_outputs)

if issues:
    print("\nPOTENTIAL SECURITY ISSUES:")
    for name, cell, label in issues:
        print(f"{name}, cell {cell}: {label}")
else:
    print("\nPASS: No obvious API keys or bearer tokens in outputs.")


NOTEBOOK OUTPUT SECURITY CHECK

01_ingest_clean.ipynb: 37 saved outputs
03_model.ipynb: 26 saved outputs
04_genai.ipynb: 21 saved outputs

Total saved outputs: 84

PASS: No obvious API keys or bearer tokens in outputs.
